# Timing ephemeris parameteters


In our measurement equation, the errors in the timing model fit are modelled as 

$$ \delta t = M \bar{\delta \epsilon}$$


Typically $M$ is an array of dimension $N_{\rm times} \times m$, where $m$ (lower case) is the number of fit parameters. $\bar{\delta \epsilon}$ is a vector of dimension $m$. 


In our case, we iterate through the data and so we have 

$$ \delta t (t) = M(t) \bar{\delta \epsilon}$$

where $M(t)$ has dimension $m$.

---

Now, we know the values of $M$ - they are given to us by the pulsar timing package (e.g. TEMPO/Enterprise). 

The goal is to "twiddle" the values of $\bar{\delta \epsilon}$.

---

In practice this is a lot of parameters to infer: $N_{\rm psr} \times m$. The value of $m$ is somewhere around 10, but varies from pulsar to pulsar. We want to avoid having to infer all these parameters if possible. Indeed, in standard pulsar timing analysis these parameters are marginalised over. We handle this by incorporating $\delta \epsilon$ into the state vector and modelling its evolution as:


$$ \frac{d \delta \epsilon}{dt} = \chi(t;\sigma)$$ 


The question is, **how to set $\sigma$?**

---



**Reflection:** is this really the best way to do this? The error in (e.g.) the RA fit is not a random walk - it is a constant value, with the variation between timesteps introduced by $M$. Perhaps then it is better to set $\sigma = 0$ and quantify the uncertainty via the P0?


Lets roll with this idea and see where we get. We require 

$$ M_{max} ^2 * P_0 \approx \Delta^2  $$

where $\Delta$ is the  1 sigma timing residual contribution from this component. 

In [1]:
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
from jax.scipy.linalg import block_diag




In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import os 
import glob
import sys 

sys.path.append('../python/argus')


from argus import data_loader
from argus import models
from argus import jax_kalman_filter
from argus import gravitational_waves


def _get_processed_residuals(directory):
    """Get the processed residuals from the data."""

    # Get all .par and .tim files in the directory
    par_files = sorted(glob.glob(directory + "*.par"))
    tim_files = sorted(glob.glob(directory + "*.tim"))

    assert len(par_files) == len(tim_files), "Mismatch between .par and .tim file counts."

    #Exclude PR J1640+2224 as it has an exponent which is to small for the OU process to be valid
    par_files = [f for f in par_files if "J1640" not in f]
    tim_files = [f for f in tim_files if "J1640" not in f]



    # Get the data
    print(f"Getting the data. Loading {len(par_files)} pulsars from {data_path}")
    pulsar_residuals, pulsar_metadata, pulsar_design_matrices = (
        data_loader.LoadWidebandPulsarData.read_multiple_par_tim(par_files, tim_files)
    )

    # Get the separation angles and compute HD correlation
    ra = pulsar_metadata["RA"].to_numpy(dtype=float)
    dec = pulsar_metadata["DEC"].to_numpy(dtype=float)
    angular_separation_matrix = data_loader.LoadWidebandPulsarData.pairwise_angular_separation(ra, dec)
    hd_correlation_matrix = gravitational_waves.hellings_downs(angular_separation_matrix)

    # Post-process the residuals
    processed_pulsar_residuals = data_loader.LoadWidebandPulsarData.post_process_residuals(pulsar_residuals)

    print("Total length of the data is ", len(processed_pulsar_residuals))
    print("Total number of pulsars is ", len(pulsar_metadata))

    return processed_pulsar_residuals, pulsar_metadata, pulsar_design_matrices,hd_correlation_matrix

In [4]:
#Get the data
data_path = "../data/IPTA_MockDataChallenge2/dataset_1b/" # https://github.com/ipta/mdc2/tree/master
processed_pulsar_residuals, pulsar_metadata, pulsar_design_matrices,hd_correlation_matrix = _get_processed_residuals(data_path)


Getting the data. Loading 32 pulsars from ../data/IPTA_MockDataChallenge2/dataset_1b/
[tempo2Util.C:396] Warning: [TIM1] Please place MODE flags in the parameter file 
[preProcess.C:158] Warning: PSR J0030+0451 uses DM2+ but does not define DM_SERIES. Assume Taylor. This has behaviour has changed since June 2020!
See https://bitbucket.org/psrsoft/tempo2/issues/27/tempo2-dm-polynomial-is-not-a-taylor

J0030+0451 205.53069593846152 183
[tempo2Util.C:401] Warning: [DUP1] duplicated warnings have been suppressed.
[preProcess.C:158] Warning: PSR J0034-0534 uses DM2+ but does not define DM_SERIES. Assume Taylor. This has behaviour has changed since June 2020!
See https://bitbucket.org/psrsoft/tempo2/issues/27/tempo2-dm-polynomial-is-not-a-taylor



J0034-0534 532.7134293952242 183
[preProcess.C:158] Warning: PSR J0218+4232 uses DM2+ but does not define DM_SERIES. Assume Taylor. This has behaviour has changed since June 2020!
See https://bitbucket.org/psrsoft/tempo2/issues/27/tempo2-dm-polynomial-is-not-a-taylor

J0218+4232 430.46105454575843 183
[preProcess.C:158] Warning: PSR J0437-4715 uses DM2+ but does not define DM_SERIES. Assume Taylor. This has behaviour has changed since June 2020!
See https://bitbucket.org/psrsoft/tempo2/issues/27/tempo2-dm-polynomial-is-not-a-taylor

J0437-4715 173.68794573751967 183
[preProcess.C:158] Warning: PSR J0613-0200 uses DM2+ but does not define DM_SERIES. Assume Taylor. This has behaviour has changed since June 2020!
See https://bitbucket.org/psrsoft/tempo2/issues/27/tempo2-dm-polynomial-is-not-a-taylor



J0613-0200 326.60056202349193 183
[preProcess.C:158] Warning: PSR J0621+1002 uses DM2+ but does not define DM_SERIES. Assume Taylor. This has behaviour has changed since June 2020!
See https://bitbucket.org/psrsoft/tempo2/issues/27/tempo2-dm-polynomial-is-not-a-taylor

J0621+1002 34.65740662142043 183
[preProcess.C:158] Warning: PSR J0711-6830 uses DM2+ but does not define DM_SERIES. Assume Taylor. This has behaviour has changed since June 2020!
See https://bitbucket.org/psrsoft/tempo2/issues/27/tempo2-dm-polynomial-is-not-a-taylor



J0711-6830 182.1172346472276 183
[preProcess.C:158] Warning: PSR J0751+1807 uses DM2+ but does not define DM_SERIES. Assume Taylor. This has behaviour has changed since June 2020!
See https://bitbucket.org/psrsoft/tempo2/issues/27/tempo2-dm-polynomial-is-not-a-taylor



J0751+1807 287.4578539950941 183
[preProcess.C:158] Warning: PSR J0900-3144 uses DM2+ but does not define DM_SERIES. Assume Taylor. This has behaviour has changed since June 2020!
See https://bitbucket.org/psrsoft/tempo2/issues/27/tempo2-dm-polynomial-is-not-a-taylor

J0900-3144 90.01184191936728 183
[preProcess.C:158] Warning: PSR J1012+5307 uses DM2+ but does not define DM_SERIES. Assume Taylor. This has behaviour has changed since June 2020!
See https://bitbucket.org/psrsoft/tempo2/issues/27/tempo2-dm-polynomial-is-not-a-taylor

J1012+5307 190.26783444155498 183
[preProcess.C:158] Warning: PSR J1022+1001 uses DM2+ but does not define DM_SERIES. Assume Taylor. This has behaviour has changed since June 2020!
See https://bitbucket.org/psrsoft/tempo2/issues/27/tempo2-dm-polynomial-is-not-a-taylor

J1022+1001 60.77944795669458 183
[preProcess.C:158] Warning: PSR J1024-0719 uses DM2+ but does not define DM_SERIES. Assume Taylor. This has behaviour has changed since June 2020!
See https://

J1603-7202 67.37658112877924 183
[preProcess.C:158] Warning: PSR J1614-2230 uses DM2+ but does not define DM_SERIES. Assume Taylor. This has behaviour has changed since June 2020!
See https://bitbucket.org/psrsoft/tempo2/issues/27/tempo2-dm-polynomial-is-not-a-taylor

J1614-2230 317.37893706872126 183
[preProcess.C:158] Warning: PSR J1643-1224 uses DM2+ but does not define DM_SERIES. Assume Taylor. This has behaviour has changed since June 2020!
See https://bitbucket.org/psrsoft/tempo2/issues/27/tempo2-dm-polynomial-is-not-a-taylor

J1643-1224 216.37333714263337 183
[preProcess.C:158] Warning: PSR J1713+0747 uses DM2+ but does not define DM_SERIES. Assume Taylor. This has behaviour has changed since June 2020!
See https://bitbucket.org/psrsoft/tempo2/issues/27/tempo2-dm-polynomial-is-not-a-taylor

J1713+0747 218.81184041715784 183
[preProcess.C:158] Warning: PSR J1730-2304 uses DM2+ but does not define DM_SERIES. Assume Taylor. This has behaviour has changed since June 2020!
See https:

J1939+2134 641.9282245822369 183
[preProcess.C:158] Warning: PSR J1944+0907 uses DM2+ but does not define DM_SERIES. Assume Taylor. This has behaviour has changed since June 2020!
See https://bitbucket.org/psrsoft/tempo2/issues/27/tempo2-dm-polynomial-is-not-a-taylor



J1944+0907 192.85651792018598 183
[preProcess.C:158] Warning: PSR J2010-1323 uses DM2+ but does not define DM_SERIES. Assume Taylor. This has behaviour has changed since June 2020!
See https://bitbucket.org/psrsoft/tempo2/issues/27/tempo2-dm-polynomial-is-not-a-taylor

J2010-1323 191.45090909263828 183
[preProcess.C:158] Warning: PSR J2124-3358 uses DM2+ but does not define DM_SERIES. Assume Taylor. This has behaviour has changed since June 2020!
See https://bitbucket.org/psrsoft/tempo2/issues/27/tempo2-dm-polynomial-is-not-a-taylor

J2124-3358 202.7938937460303 183
[preProcess.C:158] Warning: PSR J2129-5721 uses DM2+ but does not define DM_SERIES. Assume Taylor. This has behaviour has changed since June 2020!
See https://bitbucket.org/psrsoft/tempo2/issues/27/tempo2-dm-polynomial-is-not-a-taylor

J2129-5721 268.3592272938659 183
[preProcess.C:158] Warning: PSR J2145-0750 uses DM2+ but does not define DM_SERIES. Assume Taylor. This has behaviour has changed since June 2020!
See https:/

J2145-0750 62.29588783738885 183
[preProcess.C:158] Warning: PSR J2229+2643 uses DM2+ but does not define DM_SERIES. Assume Taylor. This has behaviour has changed since June 2020!
See https://bitbucket.org/psrsoft/tempo2/issues/27/tempo2-dm-polynomial-is-not-a-taylor

J2229+2643 335.8162081968717 183
[preProcess.C:158] Warning: PSR J2317+1439 uses DM2+ but does not define DM_SERIES. Assume Taylor. This has behaviour has changed since June 2020!
See https://bitbucket.org/psrsoft/tempo2/issues/27/tempo2-dm-polynomial-is-not-a-taylor

J2317+1439 290.25460366486976 183
Total length of the data is  5856
Total number of pulsars is  32


Also get some injections:

In [5]:
import json

# Load the noise parameters from the json file
with open("../data/IPTA_MockDataChallenge2/group1_psr_noise.json", "r") as f:
    noise_params = json.load(f)

# Extract EFAC and EQUAD values for each pulsar
efac_values = []
equad_values = []

for psr in noise_params:

    if  "J1640" not in psr:
        efac_values.append(noise_params[psr]["efac"])
        equad_values.append(10**noise_params[psr]["equad"]) # Convert from log10 to linear

# Convert to JAX arrays
efac_array = jnp.array(efac_values)
equad_array = jnp.array(equad_values)


assert len(efac_array) == len(equad_array) == len(pulsar_metadata)

2025-04-17 11:43:03.168893: W external/xla/xla/service/platform_util.cc:211] unable to create StreamExecutor for CUDA:0: : CUDA_ERROR_OUT_OF_MEMORY: out of memory


RuntimeError: Unable to initialize backend 'cuda': INTERNAL: no supported devices found for platform CUDA (you may need to uninstall the failing plugin package, or set JAX_PLATFORMS=cpu to skip this backend.)

In [6]:
import pandas as pd
df = pd.read_pickle('approximate_spin_injections.pkl')
condition = df['psr'] != 'J1640+2224'

# 2. Use the condition to select rows and create a new DataFrame
df_filtered = df[condition]

In [7]:
sigma_p_injected = df_filtered['optimal_sigma'].values
gamma_p_injected = df_filtered['optimal_gamma'].values

assert len(sigma_p_injected) == len(gamma_p_injected) == len(pulsar_metadata)

## Now look at defining P0 for the $\delta \epsilon$ terms 

In [8]:
def _initialize_kalman_filter(nx,Npsr,P_eps):

    """
    Specify the initial state vector x0 and the covariance matrix P0 for the Kalman filter.
    """

    # Initialize the JAX Kalman Filter
    x0 = jnp.zeros(nx) # Initial state vector. δφ=0,δf=0, etc. As all the states are effecitvely perturbations, this is a reasonable guess.


    #Initialize the covariance matrices
    P_GW = jnp.eye(Npsr * 2)
    P_GW = P_GW.at[1::2, 1::2].multiply(1e-20) # All the odd diagonal elements, (1,1), (3,3) etc. are set to 1e-12
    P_GW = P_GW.at[0::2, 0::2].multiply(1e-25) # All the even diagonal elements, (0,0), (2,2) etc. are set to 1e-18


    P_spin = jnp.eye(Npsr * 2)
    P_spin = P_spin.at[1::2, 1::2].multiply(1e-12) # All the odd diagonal elements, (1,1), (3,3) etc. are set to 1e-12
    P_spin = P_spin.at[0::2, 0::2].multiply(1e-20) # All the even diagonal elements, (0,0), (2,2) etc. are set to 1e-18


    P0 = block_diag(P_GW, P_spin, np.diag(P_eps))

    

    return x0, P0

In [ ]:
import numpy as np 

from flax import struct
import numpyro.distributions as dist


@struct.dataclass
class Parameters:

    """Define a struct to store the parameters of the Kalman filter model"""
    
    #GW parameters
    γa: float  # s⁻¹
    ha: float  # GWB amplitude

    #Pulsar parameters for the OU process
    γp: jnp.ndarray  # Pulsar-specific gamma values
    σp: jnp.ndarray  # Pulsar-specific sigma values 

    #Measurement noise parameters
    EFAC: jnp.ndarray  # Error factors
    EQUAD: jnp.ndarray # Extra quadrature noise



params = Parameters(
    #GW parameters
    γa=2.7e-2*2*np.pi*1e-8,
    ha=1e-12,

    #Spin parameters
    γp=gamma_p_injected,
    σp=sigma_p_injected,

    #Measurement noise parameters
    EFAC=efac_array,
    EQUAD=equad_array
)


import numpy as np 


#Calculate P0 based on the maximum value of the design matrix, and a delta tolerance
model = models.StochasticGWBackgroundModel(pulsar_metadata, hd_correlation_matrix, pulsar_design_matrices)
delta = 1e-3 #milliseconds
P0 = [delta**2  / np.max(pulsar_design_matrices[i],axis=0)**2 for i in range(len(pulsar_design_matrices))]


#Check that the dimensions are correct
for i in range(len(P0)):
    assert len(P0[i]) == model.M[i]

P0 = np.concatenate(P0)
assert len(P0) == model.M_sum


#Initialize the model


x_init,P_init = _initialize_kalman_filter(model.nx,model.Npsr,P0) #this could go inside the model class....

KF = jax_kalman_filter.JaxScalarKalmanFilter(
    model=model, 
    observations=processed_pulsar_residuals, 
    x0=x_init, 
    P0=P_init
)


print("Starting likelihood calculation")
ll = KF.get_likelihood(params)
print(ll)

/fred/oz022/tkimpson/conda_envs/Argus/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NameError: name 'jnp' is not defined